# Music Rating Prediction with Collaborative Filtering

## Business use case

Music platforms need to personalize large catalogs from sparse feedback. Collaborative filtering can infer likely preferences from the pattern of ratings across listeners and songs without requiring hand-built content features.

## Objective

The notebook constructs a user-song rating matrix, withholds observed ratings for validation, and applies SoftImpute low-rank matrix completion. Predictions are benchmarked against the average training rating.

## Result in context

The stored run reports **0.2789 out-of-sample R²** and **0.4789 in-sample R²**. The model outperforms the mean-rating baseline on unseen ratings.


## Step 1 — Load the sparse ratings data

The dataset is read as user ID, song ID, and rating triplets. This long format is converted into the matrix representation needed for collaborative filtering.


In [3]:
# Import necessary packages
!pip install fancyimpute
import sys
sys.path.append('/collaborativeFiltering/')
import os
os.chdir("/collaborativeFiltering/")
import numpy as np
import pandas as pd
from fancyimpute import BiScaler
from soft_impute import SoftImpute
from functionsCF import GenerateTrainingSet

In [5]:
rating = pd.read_csv('/collaborativeFiltering/MusicRatings.csv',sep=',').values
print(rating[:5, :])

[[5.26000000e+02 8.00000000e+01 1.47712125e+00]
 [1.40300000e+03 5.40000000e+01 2.20411998e+00]
 [5.56000000e+02 8.00000000e+01 1.30103000e+00]
 [1.03600000e+03 5.40000000e+01 1.47712125e+00]
 [2.35200000e+03 8.00000000e+01 1.30103000e+00]]


## Step 2 — Create the user-song matrix

Unique song IDs are remapped to contiguous column positions. Known ratings populate an otherwise missing matrix, preserving the sparsity pattern of the original feedback.


In [6]:
matrix_incomplete=np.zeros((len(np.unique(rating[:,0])), len(np.unique(rating[:,1]))))

In [7]:
usedID = np.unique(rating[:,1])
for i in range(len(rating[:,1])):
    rating[:,1][i] = np.where(usedID==rating[:,1][i])[0][0] + 1

In [8]:
matrix_incomplete[:] = np.nan
indices = np.array(rating[:,0]-1).astype(int),np.array(rating[:,1]-1).astype(int)
matrix_incomplete[indices] = rating[:,2]

## Step 3 — Create training and validation rating sets

Observed ratings are split into training and validation index pairs. Only the training positions are exposed to the matrix-completion algorithm.


In [9]:
train_indices, validation_indices = GenerateTrainingSet(rating[:,0],rating[:,1],0.8)
matrix_train = matrix_incomplete.copy()
matrix_train[:] = np.nan
matrix_train[train_indices] = matrix_incomplete[train_indices]

## Step 4 — Transform the training matrix

`BiScaler` prepares the sparse matrix for the low-rank completion step. The same fitted transform is later inverted so predictions return to the original rating scale.


In [10]:
biscaler = BiScaler(scale_rows = False, scale_columns = False, max_iters=50, verbose=False)
matrix_train_normalized = biscaler.fit_transform(matrix_train)

## Step 5 — Fit SoftImpute and recover missing ratings

SoftImpute is configured with six latent components. The completed matrix represents the model's estimate of how each listener would rate songs that were withheld or never observed.


In [11]:
softImpute = SoftImpute(J = 6, maxit = 200, random_seed = 2033, verbose = False)

In [12]:
# Run the softImpute model on the normalized training set
# Call the output matrix_train_softImpute
matrix_train_softImpute = softImpute.fit(matrix_train_normalized)
matrix_train_filled_normalized = matrix_train_softImpute.predict(matrix_train_normalized, copyto = False)
matrix_train_filled = biscaler.inverse_transform(matrix_train_filled_normalized)


## Step 6 — Establish the baseline and evaluate generalization

The average training rating is used as a simple baseline. Model and baseline MSE are compared on both training and validation indices to compute R².


In [13]:
train_average = np.average(matrix_train[train_indices])

In [14]:
validation_mse= ((matrix_train_filled[validation_indices] - matrix_incomplete[validation_indices])**2).mean()
training_mse=((matrix_train_filled[train_indices] - matrix_incomplete[train_indices])**2).mean()
validation_mse_baseline=((train_average-matrix_incomplete[validation_indices])**2).mean()
training_mse_baseline=((train_average-matrix_incomplete[train_indices])**2).mean()
print("out-of-sample R2: %.4f, in-sample R2: %.4f." % (1 - validation_mse / validation_mse_baseline, 1 - training_mse / training_mse_baseline))

out-of-sample R2: 0.2789, in-sample R2: 0.4789.


## Step 7 — Inspect an individual prediction

A single completed matrix entry is printed as a sanity check that the model produces a concrete user-song score on the original scale.


In [15]:
print("After matrix completion =", matrix_train_filled[100, 16])

After matrix completion = 1.889684596429213


## Technical conclusions

The stored evaluation gives **0.2789 out-of-sample R²** and **0.4789 in-sample R²**. The positive validation R² confirms that the low-rank model improves over a global-average predictor, while the gap to training performance could suggest room for better rank selection and regularization.

For a recommendation product, rating prediction is only part of the problem. I would add top-N ranking metrics, coverage, diversity, and time-based evaluation to understand whether the model improves discovery rather than only squared error.

## Business conclusions

The experiment supports the idea that listener behavior contains enough shared structure to personalize a sparse catalog. A production system could use the completed scores to generate candidates, then combine them with freshness, popularity, editorial constraints, and exploration.

## Limitations and next steps

Cold-start users and songs are not addressed. The next iteration should add side information or hybrid features and test recommendations using an interaction-oriented metric.
